# Notebook Comentado — Case Citrus Analytics

## 1. Importação das bibliotecas

In [31]:
# =========================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
#
# pandas:
# Biblioteca utilizada para manipulação e análise
# de dados tabulares.
#
# numpy:
# Biblioteca utilizada para operações numéricas
# e tratamento de valores nulos.
# =========================================================

import pandas as pd
import numpy as np

## 2. Configuração de visualização do Pandas

In [32]:
# =========================================================
# CONFIGURAÇÕES DE VISUALIZAÇÃO DO PANDAS
#
# display.max_columns:
# Permite visualizar todas as colunas do DataFrame.
#
# display.max_rows:
# Define limite de linhas exibidas nas visualizações.
# =========================================================

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 3. Leitura do dataset bruto

In [33]:
# =========================================================
# LEITURA DO DATASET BRUTO
#
# Arquivo:
# producao_citros_bruto.csv
#
# O dataset contém informações de produção de
# citros no Brasil entre 2015 e 2023.
# =========================================================

df = pd.read_csv('../data/producao_citros_bruto.csv')

## 4. Visualização inicial do dataset

In [34]:
# =========================================================
# VISUALIZAÇÃO INICIAL DO DATAFRAME
#
# Objetivo:
# Validar estrutura, colunas e primeiros registros.
# =========================================================

df.head()

,ano,cod_produto,produto,uf,estado,municipio,qtd_produzida_ton,area_colhida_ha,valor_producao_reais
0,2015,41151,Laranja,SP,São Paulo,araraquara,59762,3040.0,30576052.0
1,2016,41151,Laranja,SP,São Paulo,Araraquara,48091,3032.0,42397746.0
2,2017,41151,Laranja,SP,São Paulo,Araraquara,51414,3440.0,51776257.0
3,2018,41151,Laranja,SP,São Paulo,Araraquara,48244,3127.0,33152480.0
4,2019,41151,Laranja,SP,São Paulo,Araraquara,63018,2934.0,39987607.0


## 5. Estrutura e tipos das colunas

In [35]:
# =========================================================
# ANÁLISE DA ESTRUTURA DO DATAFRAME
#
# Objetivo:
# Identificar:
# - Quantidade de registros
# - Tipos das colunas
# - Valores nulos
# - Possíveis inconsistências
# =========================================================

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ano                   3600 non-null   int64  
 1   cod_produto           3600 non-null   int64  
 2   produto               3600 non-null   str    
 3   uf                    3600 non-null   str    
 4   estado                3600 non-null   str    
 5   municipio             3600 non-null   str    
 6   qtd_produzida_ton     3484 non-null   str    
 7   area_colhida_ha       3438 non-null   float64
 8   valor_producao_reais  3478 non-null   float64
dtypes: float64(2), int64(2), str(5)
memory usage: 253.3 KB


## 6. Identificação de valores inválidos

In [36]:
# =========================================================
# IDENTIFICAÇÃO DE VALORES "-"
#
# Objetivo:
# Verificar colunas que possuem valores inválidos
# representados como string.
# =========================================================

(df == '-').sum()

ano                      0
cod_produto              0
produto                  0
uf                       0
estado                   0
municipio                0
qtd_produzida_ton       97
area_colhida_ha          0
valor_producao_reais     0
dtype: int64

## 7. Criação da cópia de trabalho

In [37]:
# =========================================================
# CRIAÇÃO DE CÓPIA DO DATAFRAME
#
# Objetivo:
# Preservar o dataset original durante o processo
# de limpeza e transformação.
# =========================================================

df_clean = df.copy()

## 8. Padronização dos municípios

In [38]:
# =========================================================
# PADRONIZAÇÃO DOS NOMES DOS MUNICÍPIOS
#
# Objetivo:
# Corrigir inconsistências de capitalização para
# facilitar integração com a API do IBGE.
# =========================================================

df_clean['municipio'] = (
    df_clean['municipio']
    .str.strip()
    .str.title()
)

## 9. Substituição de valores inválidos

In [39]:
# =========================================================
# SUBSTITUIÇÃO DE VALORES INVÁLIDOS
#
# Objetivo:
# Converter valores "-" para NaN.
# =========================================================

df_clean = df_clean.replace('-', np.nan)

## 10. Conversão das colunas numéricas

In [40]:
# =========================================================
# CONVERSÃO DAS COLUNAS NUMÉRICAS
#
# Objetivo:
# Garantir tipos corretos para cálculos e análises.
# =========================================================

colunas_numericas = [
    'qtd_produzida_ton',
    'area_colhida_ha',
    'valor_producao_reais'
]

for col in colunas_numericas:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors='coerce'
    )

## 11. Validação da estrutura após limpeza

In [41]:
# =========================================================
# VALIDAÇÃO DOS TIPOS DAS COLUNAS
#
# Objetivo:
# Confirmar conversão correta das colunas numéricas.
# =========================================================

df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ano                   3600 non-null   int64  
 1   cod_produto           3600 non-null   int64  
 2   produto               3600 non-null   str    
 3   uf                    3600 non-null   str    
 4   estado                3600 non-null   str    
 5   municipio             3600 non-null   str    
 6   qtd_produzida_ton     3387 non-null   float64
 7   area_colhida_ha       3438 non-null   float64
 8   valor_producao_reais  3478 non-null   float64
dtypes: float64(3), int64(2), str(4)
memory usage: 253.3 KB


## 12. Identificação de registros inválidos

In [42]:
# =========================================================
# IDENTIFICAÇÃO DE REGISTROS INVÁLIDOS
#
# Regras de negócio:
# - Área colhida deve ser maior que zero
# - Quantidade produzida deve ser maior que zero
# =========================================================

registros_invalidos = df_clean[
    (df_clean['area_colhida_ha'] <= 0) |
    (df_clean['qtd_produzida_ton'] <= 0)
]

registros_invalidos.shape

(0, 9)

## 13. Remoção de registros inválidos

In [43]:
# =========================================================
# REMOÇÃO DE REGISTROS INVÁLIDOS
#
# Objetivo:
# Manter apenas registros válidos para análise.
# =========================================================

df_clean = df_clean[
    (df_clean['area_colhida_ha'] > 0) &
    (df_clean['qtd_produzida_ton'] > 0)
]

## 14. Criação da métrica de produtividade

In [44]:
# =========================================================
# CRIAÇÃO DA MÉTRICA DE PRODUTIVIDADE
#
# Fórmula:
# produtividade_ton_ha =
# qtd_produzida_ton / area_colhida_ha
# =========================================================

df_clean['produtividade_ton_ha'] = (
    df_clean['qtd_produzida_ton'] /
    df_clean['area_colhida_ha']
)

## 15. Estatísticas descritivas

In [45]:
# =========================================================
# ESTATÍSTICAS DESCRITIVAS
#
# Objetivo:
# Validar distribuição dos dados após limpeza.
# =========================================================

df_clean.describe()

,ano,cod_produto,qtd_produzida_ton,area_colhida_ha,valor_producao_reais,produtividade_ton_ha
count,3234.000000,3234.000000,3234.000000,3234.000000,3.124000e+03,3234.000000
mean,2019.000309,41160.573593,6781.592455,669.541744,6.720775e+06,10.452548
std,2.582688,9.551349,14181.420187,886.756646,1.252624e+07,10.017067
min,2015.000000,41151.000000,105.000000,45.000000,8.035900e+04,0.392771
25%,2017.000000,41155.000000,1154.000000,191.000000,1.303647e+06,3.895908
50%,2019.000000,41160.000000,2029.500000,364.000000,2.666222e+06,7.305758
75%,2021.000000,41176.000000,7293.750000,789.750000,7.511659e+06,13.939570
max,2023.000000,41176.000000,134485.000000,7413.000000,1.371068e+08,85.108911


## 16. Leitura da dimensão de municípios

In [46]:
# =========================================================
# LEITURA DA DIMENSÃO DE MUNICÍPIOS
#
# Objetivo:
# Carregar dados enriquecidos da API do IBGE.
# =========================================================

df_municipios = pd.read_csv('../data/dim_municipio.csv')

# Visualização inicial

df_municipios.head()

,cod_municipio_ibge,nome_municipio,uf,mesorregiao,regiao_brasil
0,1200013,Acrelândia,AC,Vale do Acre,Norte
1,1200054,Assis Brasil,AC,Vale do Acre,Norte
2,1200104,Brasiléia,AC,Vale do Acre,Norte
3,1200138,Bujari,AC,Vale do Acre,Norte
4,1200179,Capixaba,AC,Vale do Acre,Norte


## 17. Estrutura da dimensão de municípios

In [47]:
# =========================================================
# ANÁLISE DA DIMENSÃO DE MUNICÍPIOS
#
# Objetivo:
# Validar estrutura e consistência da dimensão.
# =========================================================

df_municipios.info()

<class 'pandas.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   cod_municipio_ibge  5571 non-null   int64
 1   nome_municipio      5571 non-null   str  
 2   uf                  5571 non-null   str  
 3   mesorregiao         5570 non-null   str  
 4   regiao_brasil       5570 non-null   str  
dtypes: int64(1), str(4)
memory usage: 217.7 KB


## 18. Padronização para integração

In [48]:
# =========================================================
# PADRONIZAÇÃO DOS NOMES DOS MUNICÍPIOS
#
# Objetivo:
# Garantir consistência entre os datasets.
# =========================================================

df_municipios['nome_municipio'] = (
    df_municipios['nome_municipio']
    .str.strip()
    .str.title()
)

df_clean['municipio'] = (
    df_clean['municipio']
    .str.strip()
    .str.title()
)

## 19. Integração entre produção e dimensão

In [49]:
# =========================================================
# INTEGRAÇÃO DOS DADOS
#
# Join realizado utilizando:
# - município
# - UF
#
# Objetivo:
# Associar código IBGE e informações regionais
# aos registros de produção.
# =========================================================

df_final = df_clean.merge(
    df_municipios,
    left_on=['municipio', 'uf'],
    right_on=['nome_municipio', 'uf'],
    how='left'
)

## 20. Validação do resultado do merge

In [50]:
# =========================================================
# VALIDAÇÃO DO RESULTADO DO MERGE
#
# Objetivo:
# Confirmar quantidade de registros após integração.
# =========================================================

print(df_final.shape)

df_final.head()

(3234, 14)


,ano,cod_produto,produto,uf,estado,municipio,qtd_produzida_ton,area_colhida_ha,valor_producao_reais,produtividade_ton_ha,cod_municipio_ibge,nome_municipio,mesorregiao,regiao_brasil
0,2015,41151,Laranja,SP,São Paulo,Araraquara,59762.0,3040.0,30576052.0,19.658553,3503208.0,Araraquara,Araraquara,Sudeste
1,2016,41151,Laranja,SP,São Paulo,Araraquara,48091.0,3032.0,42397746.0,15.861148,3503208.0,Araraquara,Araraquara,Sudeste
2,2017,41151,Laranja,SP,São Paulo,Araraquara,51414.0,3440.0,51776257.0,14.945930,3503208.0,Araraquara,Araraquara,Sudeste
3,2018,41151,Laranja,SP,São Paulo,Araraquara,48244.0,3127.0,33152480.0,15.428206,3503208.0,Araraquara,Araraquara,Sudeste
4,2019,41151,Laranja,SP,São Paulo,Araraquara,63018.0,2934.0,39987607.0,21.478528,3503208.0,Araraquara,Araraquara,Sudeste


## 21. Verificação de registros sem correspondência

In [51]:
# =========================================================
# VERIFICAÇÃO DE REGISTROS SEM MATCH
#
# Objetivo:
# Identificar municípios que não encontraram
# correspondência na dimensão do IBGE.
# =========================================================

sem_match = df_final[
    df_final['cod_municipio_ibge'].isnull()
]

print(f'Registros sem correspondência: {sem_match.shape[0]}')

sem_match[['municipio', 'uf']].drop_duplicates()

Registros sem correspondência: 32


,municipio,uf
1009,Petrolina,BA


## 22. Criação da tabela fato

In [52]:
# =========================================================
# CRIAÇÃO DA TABELA FATO
#
# Objetivo:
# Estruturar tabela analítica principal do modelo.
# =========================================================

fato_producao = df_final[[
    'cod_municipio_ibge',
    'cod_produto',
    'ano',
    'qtd_produzida_ton',
    'area_colhida_ha',
    'valor_producao_reais',
    'produtividade_ton_ha'
]]

## 23. Validação da tabela fato

In [53]:
# =========================================================
# VALIDAÇÃO DA TABELA FATO
#
# Objetivo:
# Confirmar estrutura final do modelo analítico.
# =========================================================

fato_producao.info()

fato_producao.head()

<class 'pandas.DataFrame'>
RangeIndex: 3234 entries, 0 to 3233
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cod_municipio_ibge    3202 non-null   float64
 1   cod_produto           3234 non-null   int64  
 2   ano                   3234 non-null   int64  
 3   qtd_produzida_ton     3234 non-null   float64
 4   area_colhida_ha       3234 non-null   float64
 5   valor_producao_reais  3124 non-null   float64
 6   produtividade_ton_ha  3234 non-null   float64
dtypes: float64(5), int64(2)
memory usage: 177.0 KB


,cod_municipio_ibge,cod_produto,ano,qtd_produzida_ton,area_colhida_ha,valor_producao_reais,produtividade_ton_ha
0,3503208.0,41151,2015,59762.0,3040.0,30576052.0,19.658553
1,3503208.0,41151,2016,48091.0,3032.0,42397746.0,15.861148
2,3503208.0,41151,2017,51414.0,3440.0,51776257.0,14.945930
3,3503208.0,41151,2018,48244.0,3127.0,33152480.0,15.428206
4,3503208.0,41151,2019,63018.0,2934.0,39987607.0,21.478528


## 24. Exportação da tabela fato

In [54]:
# =========================================================
# EXPORTAÇÃO DA TABELA FATO
#
# Objetivo:
# Salvar tabela final para consumo no Power BI.
# =========================================================

fato_producao.to_csv(
    '../data/fato_producao.csv',
    index=False
)

print('Tabela fato_producao.csv salva com sucesso!')

Tabela fato_producao.csv salva com sucesso!


## 25. Conclusão da etapa de engenharia de dados

In [55]:
# =========================================================
# CONCLUSÃO DA ETAPA
#
# Entregas realizadas:
# - Limpeza do dataset bruto
# - Tratamento de inconsistências
# - Integração com API do IBGE
# - Criação da dimensão de municípios
# - Construção da tabela fato
# - Exportação dos arquivos analíticos
# =========================================================